# LAN Chat and File-Sharing Application Using Python Socket Programming

**Course:** ICS 214 - IT Workshop III  
**Student:** Divya Ramteke  
**Roll Number:** 2025BCS0161  
**Department:** CSE

## Project summary

This project demonstrates a local-network group chat application in Python. A central TCP server manages client connections, broadcasts messages, coordinates file transfers, stores chat history in SQLite, and verifies file-transfer data using SHA-256. Clients use a small Tkinter graphical interface.

The main application consists of two Python programs: `server.py` and `client.py`. No external dataset or third-party Python package is required.

## Objectives

- Demonstrate client-server communication using TCP sockets.
- Allow multiple users to exchange text messages in one chat room.
- Transfer files in chunks and verify file integrity using SHA-256.
- Use threads so multiple connected users can communicate concurrently.
- Save chat and file-sharing records in an SQLite database.
- Handle common connection and file-transfer errors.

## System architecture

```text
                    +-----------------------+
                    | Python Chat Server    |
                    | TCP port 5000         |
                    | - connection handling |
                    | - message broadcast    |
                    | - file transfer relay  |
                    | - SQLite chat history  |
                    +-----------+-----------+
                                |
              +-----------------+-----------------+
              |                 |                 |
       +------+-------+  +------+-------+  +------+-------+
       | Client A     |  | Client B     |  | Client C ... |
       | Tkinter GUI  |  | Tkinter GUI  |  | Tkinter GUI  |
       +--------------+  +--------------+  +--------------+
```

### Message and file flow

1. A user enters the server IP address and a unique username.
2. The client opens a TCP connection to port `5000` and sends its username.
3. The server returns the connection status and recent chat history.
4. Chat messages are relayed to the connected clients and stored in SQLite.
5. A shared file is split into 64 KB chunks. The server checks the declared size and SHA-256 digest, and recipients independently verify the received file.

The server is configured to accept up to **100 registered clients**. This setting does not guarantee a particular real-Wi-Fi performance level; capacity depends on the server computer, router, and usage.

## Technology stack

| Technology | Purpose |
|---|---|
| Python | Main language |
| `socket` / TCP | Network communication |
| `threading` | Concurrent client handling and background file sending |
| Tkinter | Client graphical interface and file picker |
| SQLite (`sqlite3`) | Local chat history database |
| `hashlib` / SHA-256 | File-integrity checking |
| `json` and `struct` | Packet metadata and message framing |

### Configuration

- Server port: `5000`
- Maximum registered clients: `100`
- Maximum file size: `25 MB`
- Text message limit: `2,000` characters
- File destination: `received_files/` beside `client.py`
- Server history database: `chat_history.db` beside `server.py`

The initial version shares a file with all other connected users, instead of selecting an individual recipient.

## Implementation: server.py

Run this cell to write the server implementation into `server.py` in the current notebook folder.

In [1]:
%%writefile server.py
"""Simple multi-user LAN chat and file-sharing server.

Run with: python server.py
The server listens on port 5000 and stores chat history in chat_history.db.
This prototype is intended for trusted local networks, not the public internet.
"""

from __future__ import annotations

import hashlib
import json
import re
import socket
import sqlite3
import struct
import threading
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any

HOST = "0.0.0.0"
PORT = 5000
MAX_CLIENTS = 100
CHUNK_SIZE = 64 * 1024
MAX_FILE_SIZE = 25 * 1024 * 1024  # 25 MB per file
MAX_CHAT_LENGTH = 2000
MAX_PACKET_SIZE = CHUNK_SIZE + 256 * 1024
DB_PATH = Path(__file__).resolve().parent / "chat_history.db"

clients: dict[socket.socket, "ClientConnection"] = {}
clients_lock = threading.RLock()
transfers: dict[str, dict[str, Any]] = {}
transfers_lock = threading.RLock()
db_lock = threading.Lock()
database = sqlite3.connect(DB_PATH, check_same_thread=False)


def timestamp() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def initialize_database() -> None:
    with db_lock:
        database.execute(
            """CREATE TABLE IF NOT EXISTS messages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                username TEXT NOT NULL,
                message TEXT NOT NULL,
                message_type TEXT NOT NULL
            )"""
        )
        database.commit()


def save_message(username: str, message: str, message_type: str = "chat") -> None:
    with db_lock:
        database.execute(
            "INSERT INTO messages (timestamp, username, message, message_type) VALUES (?, ?, ?, ?)",
            (timestamp(), username, message, message_type),
        )
        database.commit()


def load_history(limit: int = 100) -> list[dict[str, str]]:
    with db_lock:
        rows = database.execute(
            "SELECT timestamp, username, message, message_type FROM messages ORDER BY id DESC LIMIT ?",
            (limit,),
        ).fetchall()
    rows.reverse()
    return [
        {"timestamp": row[0], "username": row[1], "message": row[2], "kind": row[3]}
        for row in rows
    ]


def _send_packet_unlocked(sock: socket.socket, header: dict[str, Any], payload: bytes) -> None:
    header_bytes = json.dumps(header, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    body_size = 4 + len(header_bytes) + len(payload)
    if body_size > MAX_PACKET_SIZE:
        raise ValueError("Packet is too large")
    sock.sendall(struct.pack("!I", body_size))
    sock.sendall(struct.pack("!I", len(header_bytes)))
    sock.sendall(header_bytes)
    if payload:
        sock.sendall(payload)


def send_packet(
    sock: socket.socket,
    header: dict[str, Any],
    payload: bytes = b"",
    lock: threading.Lock | threading.RLock | None = None,
) -> None:
    if lock is None:
        _send_packet_unlocked(sock, header, payload)
    else:
        with lock:
            _send_packet_unlocked(sock, header, payload)


def recv_exact(sock: socket.socket, count: int) -> bytes:
    parts: list[bytes] = []
    remaining = count
    while remaining:
        try:
            part = sock.recv(remaining)
        except socket.timeout:
            # Keep waiting through idle timeouts without losing partial packet data.
            continue
        if not part:
            raise ConnectionError("The connection was closed")
        parts.append(part)
        remaining -= len(part)
    return b"".join(parts)


def recv_packet(sock: socket.socket) -> tuple[dict[str, Any], bytes]:
    body_size = struct.unpack("!I", recv_exact(sock, 4))[0]
    if body_size < 4 or body_size > MAX_PACKET_SIZE:
        raise ValueError("Invalid or oversized packet")
    header_size = struct.unpack("!I", recv_exact(sock, 4))[0]
    if header_size < 2 or header_size > body_size - 4 or header_size > 256 * 1024:
        raise ValueError("Invalid packet header")
    header = json.loads(recv_exact(sock, header_size).decode("utf-8"))
    if not isinstance(header, dict):
        raise ValueError("Packet header must be an object")
    payload_size = body_size - 4 - header_size
    payload = recv_exact(sock, payload_size) if payload_size else b""
    return header, payload


def clean_username(value: Any) -> str:
    if not isinstance(value, str):
        return ""
    value = re.sub(r"[\x00-\x1f\x7f]", "", value).strip()
    return value[:24]


def clean_filename(value: Any) -> str:
    if not isinstance(value, str):
        return "shared_file"
    value = value.replace("\\", "/").split("/")[-1]
    value = re.sub(r"[\x00-\x1f\x7f]", "", value).strip()
    return value[:180] or "shared_file"


@dataclass
class ClientConnection:
    sock: socket.socket
    address: tuple[Any, ...]
    username: str
    send_lock: threading.Lock = field(default_factory=threading.Lock)


def send_to(client: ClientConnection, header: dict[str, Any], payload: bytes = b"") -> None:
    send_packet(client.sock, header, payload, client.send_lock)


def broadcast_packet(
    header: dict[str, Any],
    payload: bytes = b"",
    exclude: ClientConnection | None = None,
) -> None:
    with clients_lock:
        recipients = list(clients.values())
    failed: list[ClientConnection] = []
    for client in recipients:
        if client is exclude:
            continue
        try:
            send_to(client, header, payload)
        except (OSError, ValueError):
            failed.append(client)
    for client in failed:
        drop_connection(client, announce=False)


def send_error(client: ClientConnection | None, message: str) -> None:
    if client is None:
        return
    try:
        send_to(client, {"type": "error", "message": message})
    except OSError:
        drop_connection(client, announce=False)


def drop_connection(client: ClientConnection, announce: bool = True) -> None:
    with clients_lock:
        was_registered = clients.get(client.sock) is client
        if was_registered:
            del clients[client.sock]

    try:
        client.sock.shutdown(socket.SHUT_RDWR)
    except OSError:
        pass
    try:
        client.sock.close()
    except OSError:
        pass

    # Also clean up if another broadcast already removed this client.
    with transfers_lock:
        abandoned = [
            (transfer_id, info)
            for transfer_id, info in transfers.items()
            if info["sender"] is client
        ]
        for transfer_id, _ in abandoned:
            transfers.pop(transfer_id, None)
    for transfer_id, _info in abandoned:
        broadcast_packet(
            {"type": "file_abort", "file_id": transfer_id, "message": "Sender disconnected before transfer completed."},
            exclude=client,
        )

    if announce and was_registered:
        broadcast_packet({"type": "system", "message": f"{client.username} disconnected.", "timestamp": timestamp()})
        print(f"[-] {client.username} disconnected from {client.address}")


def register_client(sock: socket.socket, address: tuple[Any, ...], requested_name: str) -> ClientConnection:
    username = clean_username(requested_name)
    if not username:
        raise ValueError("Enter a username containing at least one visible character.")

    # Acquire the connection's send lock before registration so its welcome and
    # history packets always arrive before any concurrent broadcast.
    client = ClientConnection(sock=sock, address=address, username=username)
    client.send_lock.acquire()
    try:
        with clients_lock:
            if len(clients) >= MAX_CLIENTS:
                raise RuntimeError(f"The server is full (limit: {MAX_CLIENTS} users).")
            if any(c.username.casefold() == username.casefold() for c in clients.values()):
                raise ValueError("That username is already in use. Please choose another.")
            clients[sock] = client

        _send_packet_unlocked(sock, {"type": "welcome", "username": username}, b"")
        _send_packet_unlocked(sock, {"type": "history", "messages": load_history()}, b"")
    except Exception:
        with clients_lock:
            if clients.get(sock) is client:
                clients.pop(sock, None)
        raise
    finally:
        client.send_lock.release()

    broadcast_packet({"type": "system", "message": f"{username} joined the chat.", "timestamp": timestamp()})
    print(f"[+] {username} connected from {address}. Users: {len(clients)}/{MAX_CLIENTS}")
    return client


def handle_chat(client: ClientConnection, header: dict[str, Any]) -> None:
    message = header.get("message", "")
    if not isinstance(message, str):
        send_error(client, "Message must be text.")
        return
    message = message.strip()
    if not message:
        return
    if len(message) > MAX_CHAT_LENGTH:
        send_error(client, f"Messages must be {MAX_CHAT_LENGTH} characters or fewer.")
        return
    time_text = timestamp()
    save_message(client.username, message, "chat")
    broadcast_packet(
        {"type": "chat", "username": client.username, "message": message, "timestamp": time_text, "kind": "chat"}
    )


def handle_file_start(client: ClientConnection, header: dict[str, Any]) -> None:
    file_id = header.get("file_id")
    filename = clean_filename(header.get("filename"))
    size = header.get("size")
    digest = header.get("sha256")
    if not isinstance(file_id, str) or not (1 <= len(file_id) <= 64):
        send_error(client, "Invalid file-transfer identifier.")
        return
    if not isinstance(size, int) or isinstance(size, bool) or size < 0 or size > MAX_FILE_SIZE:
        send_error(client, "Files must be 25 MB or smaller.")
        return
    if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-fA-F]{64}", digest):
        send_error(client, "Invalid SHA-256 checksum.")
        return
    with transfers_lock:
        if file_id in transfers:
            send_error(client, "That file-transfer identifier is already active.")
            return
        transfers[file_id] = {
            "sender": client,
            "filename": filename,
            "size": size,
            "sha256": digest.lower(),
            "received": 0,
            "hasher": hashlib.sha256(),
        }
    broadcast_packet(
        {
            "type": "file_start",
            "file_id": file_id,
            "filename": filename,
            "size": size,
            "sha256": digest.lower(),
            "sender": client.username,
        },
        exclude=client,
    )


def handle_file_chunk(client: ClientConnection, header: dict[str, Any], payload: bytes) -> None:
    file_id = header.get("file_id")
    if not isinstance(file_id, str):
        send_error(client, "Invalid file-transfer identifier.")
        return
    with transfers_lock:
        info = transfers.get(file_id)
        if info is None or info["sender"] is not client:
            send_error(client, "Unknown file transfer. Please try sharing the file again.")
            return
        if not payload or len(payload) > CHUNK_SIZE or info["received"] + len(payload) > info["size"]:
            transfers.pop(file_id, None)
            invalid = True
        else:
            info["received"] += len(payload)
            info["hasher"].update(payload)
            invalid = False
    if invalid:
        broadcast_packet({"type": "file_abort", "file_id": file_id, "message": "Invalid file data received."}, exclude=client)
        send_error(client, "File data did not match the declared size.")
        return
    broadcast_packet({"type": "file_chunk", "file_id": file_id}, payload, exclude=client)


def handle_file_end(client: ClientConnection, header: dict[str, Any]) -> None:
    file_id = header.get("file_id")
    if not isinstance(file_id, str):
        send_error(client, "Invalid file-transfer identifier.")
        return
    with transfers_lock:
        info = transfers.get(file_id)
        if info is None or info["sender"] is not client:
            send_error(client, "Unknown file transfer.")
            return
        transfers.pop(file_id, None)
        size_ok = info["received"] == info["size"]
        hash_ok = info["hasher"].hexdigest() == info["sha256"]
    if not size_ok or not hash_ok:
        message = "File transfer failed its size or SHA-256 integrity check."
        broadcast_packet({"type": "file_abort", "file_id": file_id, "message": message}, exclude=client)
        send_error(client, message)
        return

    broadcast_packet({"type": "file_end", "file_id": file_id, "sha256": info["sha256"]}, exclude=client)
    display_message = f"Shared file: {info['filename']} ({info['size']} bytes)"
    time_text = timestamp()
    save_message(client.username, display_message, "file")
    broadcast_packet(
        {
            "type": "chat",
            "username": client.username,
            "message": display_message,
            "timestamp": time_text,
            "kind": "file",
        }
    )
    print(f"[FILE] {client.username} shared {info['filename']} ({info['size']} bytes)")


def handle_client(sock: socket.socket, address: tuple[Any, ...]) -> None:
    client: ClientConnection | None = None
    try:
        first_header, first_payload = recv_packet(sock)
        if first_payload or first_header.get("type") != "hello":
            send_packet(sock, {"type": "error", "message": "Send a valid username to connect."})
            return
        try:
            client = register_client(sock, address, first_header.get("username", ""))
        except (ValueError, RuntimeError) as exc:
            try:
                send_packet(sock, {"type": "error", "message": str(exc)})
            except OSError:
                pass
            return

        while True:
            header, payload = recv_packet(sock)
            packet_type = header.get("type")
            if packet_type == "chat":
                if payload:
                    send_error(client, "Chat packets cannot contain file data.")
                else:
                    handle_chat(client, header)
            elif packet_type == "file_start":
                if payload:
                    send_error(client, "File metadata packet cannot contain file data.")
                else:
                    handle_file_start(client, header)
            elif packet_type == "file_chunk":
                handle_file_chunk(client, header, payload)
            elif packet_type == "file_end":
                if payload:
                    send_error(client, "File-end packet cannot contain file data.")
                else:
                    handle_file_end(client, header)
            elif packet_type == "file_abort":
                file_id = header.get("file_id")
                with transfers_lock:
                    info = transfers.get(file_id) if isinstance(file_id, str) else None
                    if info is not None and info["sender"] is client:
                        transfers.pop(file_id, None)
                    else:
                        info = None
                if info is not None:
                    broadcast_packet(
                        {"type": "file_abort", "file_id": file_id, "message": "Sender cancelled the transfer."},
                        exclude=client,
                    )
            else:
                send_error(client, "Unknown request type.")
    except (ConnectionError, OSError, ValueError, json.JSONDecodeError) as exc:
        if client is not None:
            print(f"[!] Connection issue with {client.username}: {exc}")
    finally:
        if client is not None:
            drop_connection(client, announce=True)
        else:
            try:
                sock.close()
            except OSError:
                pass


def main() -> None:
    initialize_database()
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        server.bind((HOST, PORT))
        server.listen(MAX_CLIENTS)
        print("=" * 56)
        print(" Simple LAN Chat and File-Sharing Server")
        print(f" Listening on {HOST}:{PORT} | maximum clients: {MAX_CLIENTS}")
        print(f" Chat history database: {DB_PATH.name}")
        print(" Press Ctrl+C to stop the server.")
        print("=" * 56)
        while True:
            client_sock, address = server.accept()
            client_sock.settimeout(15.0)
            thread = threading.Thread(target=handle_client, args=(client_sock, address), daemon=True)
            thread.start()
    except KeyboardInterrupt:
        print("\nStopping server...")
    except OSError as exc:
        print(f"Could not start server: {exc}")
        print("Check whether port 5000 is already in use.")
    finally:
        server.close()
        with clients_lock:
            active = list(clients.values())
        for client in active:
            drop_connection(client, announce=False)
        with db_lock:
            database.close()
        print("Server stopped.")


if __name__ == "__main__":
    main()


Overwriting server.py


## Implementation: client.py

Run this cell to write the Tkinter client implementation into `client.py` in the current notebook folder.

In [2]:
%%writefile client.py
"""Simple multi-user LAN chat and file-sharing client with a Tkinter GUI.

Run with: python client.py
Connect to 127.0.0.1 for local testing, or enter the server computer's LAN IP.
"""

from __future__ import annotations

import hashlib
import json
import os
import queue
import re
import socket
import struct
import threading
import uuid
from pathlib import Path
from tkinter import END, DISABLED, NORMAL, Tk, StringVar, filedialog, messagebox
from tkinter import ttk
from tkinter.scrolledtext import ScrolledText
from typing import Any

PORT = 5000
CHUNK_SIZE = 64 * 1024
MAX_FILE_SIZE = 25 * 1024 * 1024
MAX_CHAT_LENGTH = 2000
MAX_PACKET_SIZE = CHUNK_SIZE + 256 * 1024
APP_DIR = Path(__file__).resolve().parent
RECEIVED_DIR = APP_DIR / "received_files"


def _send_packet_unlocked(sock: socket.socket, header: dict[str, Any], payload: bytes) -> None:
    header_bytes = json.dumps(header, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    body_size = 4 + len(header_bytes) + len(payload)
    if body_size > MAX_PACKET_SIZE:
        raise ValueError("Packet is too large")
    sock.sendall(struct.pack("!I", body_size))
    sock.sendall(struct.pack("!I", len(header_bytes)))
    sock.sendall(header_bytes)
    if payload:
        sock.sendall(payload)


def send_packet(
    sock: socket.socket,
    header: dict[str, Any],
    payload: bytes = b"",
    lock: threading.Lock | None = None,
) -> None:
    if lock is None:
        _send_packet_unlocked(sock, header, payload)
    else:
        with lock:
            _send_packet_unlocked(sock, header, payload)


def recv_exact(sock: socket.socket, count: int) -> bytes:
    parts: list[bytes] = []
    remaining = count
    while remaining:
        try:
            part = sock.recv(remaining)
        except socket.timeout:
            continue
        if not part:
            raise ConnectionError("The server closed the connection")
        parts.append(part)
        remaining -= len(part)
    return b"".join(parts)


def recv_packet(sock: socket.socket) -> tuple[dict[str, Any], bytes]:
    body_size = struct.unpack("!I", recv_exact(sock, 4))[0]
    if body_size < 4 or body_size > MAX_PACKET_SIZE:
        raise ValueError("Invalid or oversized packet")
    header_size = struct.unpack("!I", recv_exact(sock, 4))[0]
    if header_size < 2 or header_size > body_size - 4 or header_size > 256 * 1024:
        raise ValueError("Invalid packet header")
    header = json.loads(recv_exact(sock, header_size).decode("utf-8"))
    if not isinstance(header, dict):
        raise ValueError("Packet header must be an object")
    payload_size = body_size - 4 - header_size
    payload = recv_exact(sock, payload_size) if payload_size else b""
    return header, payload


def safe_filename(value: str) -> str:
    value = value.replace("\\", "/").split("/")[-1]
    value = re.sub(r"[\x00-\x1f\x7f]", "", value).strip()
    return value[:180] or "received_file"


def unique_destination(folder: Path, filename: str) -> Path:
    folder.mkdir(parents=True, exist_ok=True)
    original = folder / safe_filename(filename)
    if not original.exists():
        return original
    stem, suffix = original.stem, original.suffix
    counter = 1
    while True:
        candidate = folder / f"{stem} ({counter}){suffix}"
        if not candidate.exists():
            return candidate
        counter += 1


class ChatClientApp:
    def __init__(self, root: Tk) -> None:
        self.root = root
        self.root.title("Simple LAN Chat and File Sharing")
        self.root.geometry("800x590")
        self.root.minsize(640, 450)

        self.sock: socket.socket | None = None
        self.send_lock = threading.Lock()
        self.events: queue.Queue[tuple[str, Any]] = queue.Queue()
        self.closing = False
        self.username = ""
        self.incoming_files: dict[str, dict[str, Any]] = {}

        self.host_var = StringVar(value="127.0.0.1")
        self.username_var = StringVar()
        self.status_var = StringVar(value="Not connected")

        self._build_ui()
        self.root.protocol("WM_DELETE_WINDOW", self.close_app)
        self.root.after(100, self._poll_events)

    def _build_ui(self) -> None:
        connection = ttk.LabelFrame(self.root, text="Connection")
        connection.pack(fill="x", padx=10, pady=(10, 5))

        ttk.Label(connection, text="Server IP:").grid(row=0, column=0, padx=(8, 4), pady=8, sticky="w")
        self.host_entry = ttk.Entry(connection, textvariable=self.host_var, width=18)
        self.host_entry.grid(row=0, column=1, padx=4, pady=8, sticky="w")

        ttk.Label(connection, text="Username:").grid(row=0, column=2, padx=(12, 4), pady=8, sticky="w")
        self.username_entry = ttk.Entry(connection, textvariable=self.username_var, width=18)
        self.username_entry.grid(row=0, column=3, padx=4, pady=8, sticky="w")

        self.connect_button = ttk.Button(connection, text="Connect", command=self.connect)
        self.connect_button.grid(row=0, column=4, padx=8, pady=8)
        connection.columnconfigure(5, weight=1)

        self.chat_log = ScrolledText(self.root, wrap="word", state=DISABLED, font=("Consolas", 10))
        self.chat_log.pack(fill="both", expand=True, padx=10, pady=5)

        compose = ttk.Frame(self.root)
        compose.pack(fill="x", padx=10, pady=(5, 5))
        self.message_var = StringVar()
        self.message_entry = ttk.Entry(compose, textvariable=self.message_var, state=DISABLED)
        self.message_entry.pack(side="left", fill="x", expand=True, padx=(0, 6))
        self.message_entry.bind("<Return>", lambda _event: self.send_message())
        self.send_button = ttk.Button(compose, text="Send", command=self.send_message, state=DISABLED)
        self.send_button.pack(side="left", padx=(0, 6))
        self.file_button = ttk.Button(compose, text="Share File", command=self.choose_file, state=DISABLED)
        self.file_button.pack(side="left")

        ttk.Label(self.root, textvariable=self.status_var, anchor="w", relief="sunken").pack(fill="x", side="bottom")

    def _log(self, message: str) -> None:
        self.chat_log.configure(state=NORMAL)
        self.chat_log.insert(END, message + "\n")
        self.chat_log.see(END)
        self.chat_log.configure(state=DISABLED)

    def connect(self) -> None:
        host = self.host_var.get().strip()
        username = self.username_var.get().strip()
        if not host or not username:
            messagebox.showwarning("Missing details", "Enter the server IP address and a username.", parent=self.root)
            return
        if len(username) > 24:
            messagebox.showwarning("Username too long", "Use a username of 24 characters or fewer.", parent=self.root)
            return
        self.connect_button.configure(state=DISABLED)
        self.status_var.set("Connecting...")
        threading.Thread(target=self._connect_worker, args=(host, username), daemon=True).start()

    def _connect_worker(self, host: str, username: str) -> None:
        sock: socket.socket | None = None
        try:
            sock = socket.create_connection((host, PORT), timeout=10)
            sock.settimeout(15.0)
            self.send_lock = threading.Lock()
            send_packet(sock, {"type": "hello", "username": username}, lock=self.send_lock)
            header, payload = recv_packet(sock)
            if payload:
                raise ValueError("Unexpected data received during connection")
            if header.get("type") != "welcome":
                raise ConnectionError(header.get("message", "The server rejected the connection."))
            self.sock = sock
            self.username = str(header.get("username", username))
            self.events.put(("connected", self.username))
            threading.Thread(target=self._receive_loop, args=(sock,), daemon=True).start()
        except Exception as exc:
            if sock is not None:
                try:
                    sock.close()
                except OSError:
                    pass
            self.events.put(("connection_failed", str(exc)))

    def _receive_loop(self, sock: socket.socket) -> None:
        reason = "Disconnected from server."
        try:
            while not self.closing:
                header, payload = recv_packet(sock)
                packet_type = header.get("type")
                if packet_type == "history":
                    self.events.put(("history", header.get("messages", [])))
                elif packet_type == "chat":
                    self.events.put(("chat", header))
                elif packet_type == "system":
                    self.events.put(("system", header.get("message", "System message")))
                elif packet_type == "error":
                    self.events.put(("error", header.get("message", "Server error")))
                elif packet_type == "file_start":
                    self._receive_file_start(header)
                elif packet_type == "file_chunk":
                    self._receive_file_chunk(header, payload)
                elif packet_type == "file_end":
                    self._receive_file_end(header)
                elif packet_type == "file_abort":
                    self._receive_file_abort(header)
        except (ConnectionError, OSError, ValueError, json.JSONDecodeError) as exc:
            reason = str(exc) or reason
        finally:
            try:
                sock.close()
            except OSError:
                pass
            self.events.put(("disconnected", reason))

    def _receive_file_start(self, header: dict[str, Any]) -> None:
        file_id = header.get("file_id")
        filename = safe_filename(str(header.get("filename", "received_file")))
        size = header.get("size", -1)
        digest = str(header.get("sha256", "")).lower()
        sender = str(header.get("sender", "Another user"))
        if not isinstance(file_id, str) or not isinstance(size, int) or size < 0 or size > MAX_FILE_SIZE:
            self.events.put(("error", "Received invalid file-transfer details."))
            return
        if not re.fullmatch(r"[0-9a-f]{64}", digest):
            self.events.put(("error", f"Invalid checksum for {filename}; transfer ignored."))
            return
        if file_id in self.incoming_files:
            self.events.put(("error", f"Duplicate file transfer ignored: {filename}"))
            return
        try:
            destination = unique_destination(RECEIVED_DIR, filename)
            handle = destination.open("wb")
            self.incoming_files[file_id] = {
                "handle": handle,
                "path": destination,
                "filename": filename,
                "expected_size": size,
                "received": 0,
                "expected_hash": digest,
                "hasher": hashlib.sha256(),
                "sender": sender,
            }
            self.events.put(("status", f"Receiving {filename} from {sender}..."))
        except OSError as exc:
            self.events.put(("error", f"Could not save {filename}: {exc}"))

    def _receive_file_chunk(self, header: dict[str, Any], payload: bytes) -> None:
        file_id = header.get("file_id")
        info = self.incoming_files.get(file_id) if isinstance(file_id, str) else None
        if info is None:
            return
        if info["received"] + len(payload) > info["expected_size"]:
            self._discard_incoming_file(file_id, "Received file was larger than expected.")
            return
        try:
            info["handle"].write(payload)
            info["hasher"].update(payload)
            info["received"] += len(payload)
        except OSError as exc:
            self._discard_incoming_file(file_id, f"Could not write received file: {exc}")

    def _receive_file_end(self, header: dict[str, Any]) -> None:
        file_id = header.get("file_id")
        info = self.incoming_files.pop(file_id, None) if isinstance(file_id, str) else None
        if info is None:
            return
        try:
            info["handle"].close()
            size_ok = info["received"] == info["expected_size"]
            hash_ok = info["hasher"].hexdigest() == info["expected_hash"] == str(header.get("sha256", "")).lower()
            if size_ok and hash_ok:
                self.events.put(("file_received", str(info["path"])))
            else:
                try:
                    info["path"].unlink(missing_ok=True)
                except OSError:
                    pass
                self.events.put(("error", f"Integrity check failed for {info['filename']}; incomplete file deleted."))
        except OSError as exc:
            self.events.put(("error", f"Could not finish receiving {info['filename']}: {exc}"))

    def _receive_file_abort(self, header: dict[str, Any]) -> None:
        file_id = header.get("file_id")
        message = str(header.get("message", "File transfer was cancelled."))
        self._discard_incoming_file(file_id, message)

    def _discard_incoming_file(self, file_id: Any, message: str) -> None:
        info = self.incoming_files.pop(file_id, None) if isinstance(file_id, str) else None
        if info is not None:
            try:
                info["handle"].close()
            except OSError:
                pass
            try:
                info["path"].unlink(missing_ok=True)
            except OSError:
                pass
            self.events.put(("status", f"Transfer stopped: {message}"))

    def send_message(self) -> None:
        sock = self.sock
        message = self.message_var.get().strip()
        if sock is None or not message:
            return
        if len(message) > MAX_CHAT_LENGTH:
            messagebox.showwarning("Message too long", f"Keep messages under {MAX_CHAT_LENGTH} characters.", parent=self.root)
            return
        try:
            send_packet(sock, {"type": "chat", "message": message}, lock=self.send_lock)
            self.message_var.set("")
        except (OSError, ValueError) as exc:
            self.events.put(("error", f"Could not send message: {exc}"))

    def choose_file(self) -> None:
        if self.sock is None:
            return
        filename = filedialog.askopenfilename(parent=self.root, title="Choose a file to share")
        if filename:
            threading.Thread(target=self._send_file_worker, args=(Path(filename), self.sock), daemon=True).start()

    def _send_file_worker(self, path: Path, sock: socket.socket) -> None:
        try:
            size = path.stat().st_size
            if size > MAX_FILE_SIZE:
                self.events.put(("error", "Files must be 25 MB or smaller."))
                return
            digest = hashlib.sha256()
            with path.open("rb") as file_handle:
                for block in iter(lambda: file_handle.read(1024 * 1024), b""):
                    digest.update(block)
            checksum = digest.hexdigest()
            transfer_id = uuid.uuid4().hex
            send_packet(
                sock,
                {
                    "type": "file_start",
                    "file_id": transfer_id,
                    "filename": path.name,
                    "size": size,
                    "sha256": checksum,
                },
                lock=self.send_lock,
            )
            sent = 0
            last_percent = -1
            with path.open("rb") as file_handle:
                while True:
                    chunk = file_handle.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    send_packet(sock, {"type": "file_chunk", "file_id": transfer_id}, chunk, self.send_lock)
                    sent += len(chunk)
                    percent = 100 if size == 0 else int((sent / size) * 100)
                    if percent != last_percent and (percent == 100 or percent // 10 != last_percent // 10):
                        self.events.put(("status", f"Sending {path.name}: {percent}%"))
                        last_percent = percent
            send_packet(sock, {"type": "file_end", "file_id": transfer_id}, lock=self.send_lock)
            self.events.put(("status", f"File sent; waiting for server verification: {path.name}"))
        except (OSError, ValueError) as exc:
            self.events.put(("error", f"Could not share {path.name}: {exc}"))

    def _poll_events(self) -> None:
        try:
            while True:
                event, value = self.events.get_nowait()
                if event == "connected":
                    self.username = str(value)
                    self.host_entry.configure(state=DISABLED)
                    self.username_entry.configure(state=DISABLED)
                    self.connect_button.configure(state=DISABLED)
                    self.message_entry.configure(state=NORMAL)
                    self.send_button.configure(state=NORMAL)
                    self.file_button.configure(state=NORMAL)
                    self.status_var.set(f"Connected as {self.username} - share files up to 25 MB")
                    self._log(f"[SYSTEM] Connected as {self.username}.")
                    self.message_entry.focus_set()
                elif event == "connection_failed":
                    self.status_var.set("Connection failed")
                    self.connect_button.configure(state=NORMAL)
                    self._log(f"[ERROR] Connection failed: {value}")
                    messagebox.showerror("Connection failed", str(value), parent=self.root)
                elif event == "disconnected":
                    if self.sock is not None:
                        self.sock = None
                    self.host_entry.configure(state=NORMAL)
                    self.username_entry.configure(state=NORMAL)
                    self.connect_button.configure(state=NORMAL)
                    self.message_entry.configure(state=DISABLED)
                    self.send_button.configure(state=DISABLED)
                    self.file_button.configure(state=DISABLED)
                    self.status_var.set("Not connected")
                    if not self.closing:
                        self._log(f"[SYSTEM] {value}")
                elif event == "history":
                    for item in value if isinstance(value, list) else []:
                        if isinstance(item, dict):
                            self._display_chat(item)
                elif event == "chat":
                    self._display_chat(value)
                elif event == "system":
                    self._log(f"[SYSTEM] {value}")
                elif event == "error":
                    self._log(f"[ERROR] {value}")
                    self.status_var.set(str(value))
                elif event == "status":
                    self.status_var.set(str(value))
                elif event == "file_received":
                    self.status_var.set(f"File received and verified: {value}")
                    self._log(f"[FILE OK] Saved and verified: {value}")
                elif event == "shutdown":
                    pass
        except queue.Empty:
            pass
        if not self.closing:
            self.root.after(100, self._poll_events)

    def _display_chat(self, item: dict[str, Any]) -> None:
        time_text = str(item.get("timestamp", ""))
        username = str(item.get("username", "Unknown"))
        message = str(item.get("message", ""))
        self._log(f"[{time_text}] {username}: {message}")

    def close_app(self) -> None:
        self.closing = True
        sock = self.sock
        self.sock = None
        if sock is not None:
            try:
                sock.shutdown(socket.SHUT_RDWR)
            except OSError:
                pass
            try:
                sock.close()
            except OSError:
                pass
        for info in list(self.incoming_files.values()):
            try:
                info["handle"].close()
            except OSError:
                pass
            try:
                info["path"].unlink(missing_ok=True)
            except OSError:
                pass
        self.incoming_files.clear()
        self.root.destroy()


def main() -> None:
    root = Tk()
    ChatClientApp(root)
    root.mainloop()


if __name__ == "__main__":
    main()


Overwriting client.py


## Verify the generated source files

Run this cell after the two `%%writefile` cells. It checks that both scripts compile. It does not start a server or open a GUI.

In [3]:
from pathlib import Path
import py_compile

project_files = [Path("server.py"), Path("client.py")]
for file_path in project_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Run the writefile cell for {file_path.name} first.")
    py_compile.compile(str(file_path), doraise=True)
    print(f"Syntax check passed: {file_path.name} ({file_path.stat().st_size:,} bytes)")
print("Both Python programs are ready to run.")

Syntax check passed: server.py (17,092 bytes)
Syntax check passed: client.py (20,250 bytes)
Both Python programs are ready to run.


## How to run the application

### First, test it on one computer

1. Open a terminal in the folder containing `server.py` and `client.py`.
2. Run `python server.py` and leave that terminal open.
3. Open another terminal in the same folder and run `python client.py`.
4. Use `127.0.0.1` as the Server IP, enter a username, and click **Connect**.
5. Run `python client.py` in another terminal to open a second client. Use a different username.
6. Send a chat message or click **Share File**. The other clients should receive it.

### Then, test across the same Wi-Fi/LAN

1. Run the server on one computer.
2. Find that computer's local IPv4 address (on Windows, use `ipconfig`).
3. On each other computer, run `client.py`, enter the server computer's IPv4 address and a unique username, then connect.
4. Allow Python through the Windows Firewall on a private network if prompted.

A guest Wi-Fi network may block communication between devices. This prototype is for trusted local testing only; messages are not encrypted and usernames are not authenticated.

## Test plan and observed checks

| Test | Result |
|---|---|
| Python syntax compilation for both programs | Passed |
| Two local clients connect and exchange a chat message | Passed in protocol smoke test |
| Binary file sent in chunks and reconstructed byte-for-byte | Passed in protocol smoke test |
| SHA-256 comparison for the test file | Passed |
| Disconnect handling | Tested in local protocol smoke test |
| Two Tkinter clients exchange chat and a file | Passed under a virtual display; recipient's file and checksum verified |
| 100 simulated local clients connect; 101st rejected | Passed for connection-capacity check |
| Performance of 100 active users on a physical Wi-Fi router | Not measured; must be tested in the intended network |
| Full manual GUI demonstration | Run on a computer with a desktop display |

The 100-client check confirms the configured connection limit under a local simulation. It is not a throughput benchmark for 100 people chatting and transferring files simultaneously.

## Limitations and future improvements

- No encryption, login passwords, or authenticated identities.
- File sharing is broadcast to other connected users rather than targeted to a selected person.
- One server process manages the connected clients; server shutdown ends active sessions.
- File size is limited to 25 MB to keep transfers manageable for the course prototype.
- Performance depends on hardware, Wi-Fi quality, firewall rules, and network client-isolation settings.

Possible future work includes private messages, transfer progress for recipients, stronger authentication and encryption, file-transfer resume, and formal load testing.

## Conclusion

The project combines basic Python networking, threading, a desktop UI, database storage, and file-integrity verification in a runnable local-network application. It is designed to demonstrate these concepts clearly while avoiding external services and third-party dependencies.